In [ ]:
pip install transformers torch

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_name="microsoft/DialoGPT-medium"
tokenizer=AutoTokenizer.from_pretrained(model_name)
model=AutoModelForCausalLM.from_pretrained(model_name)
print("Model loaded successfully!\n")

def chatbot():
  print("Chatbot: Hello! I am your AI assistant. How can I help you today?")

  chat_history_ids=None

  while True:
    user_input=input("User:")

    if user_input.lower() in ["exit","quit"]:
      print("Chatbot: Goodbye! Have a great day!")
      break

    if "artificial intelligence" in user_input.lower():
      print("Chatbot: Artificial Intelligence refers to the simulation of human intelligence in machines that can perform tasks like learning and problem solving.")
      print()
      continue

    if "python" in user_input.lower():
      print("Chatbot: Python was created by Guido van Rossum and released in 1991.")
      print()
      continue

    new_input_ids=tokenizer.encode(user_input+tokenizer.eos_token,return_tensors='pt')

    if chat_history_ids is not None:
      bot_input_ids=torch.cat([chat_history_ids,new_input_ids],dim=-1)

    else:
      bot_input_ids=new_input_ids

    bot_input_ids=bot_input_ids[:, -512:]
    attention_mask=torch.ones(bot_input_ids.shape,dtype=torch.long)

    output_ids=model.generate(
        bot_input_ids,
        attention_mask=attention_mask,
        max_length=bot_input_ids.shape[-1] + 50,
        pad_token_id=tokenizer.eos_token_id,
        do_sample=True,
        top_k=50,
        top_p=0.95,
        temperature=0.7
    )
    chat_history_ids=output_ids

    response=tokenizer.decode(output_ids[:,bot_input_ids.shape[-1]:][0],skip_special_tokens=True)
    print("Chatbot:",response)
    print()
chatbot()